In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
crime = pd.read_csv('/content/drive/MyDrive/Dataset/Preprocessed/crime_grade.csv')
rent_deposit = pd.read_csv('/content/drive/MyDrive/Dataset/Preprocessed/rent_deposit.csv')
rent_monthly = pd.read_csv('/content/drive/MyDrive/Dataset/Preprocessed/rent_montly.csv')
commercial = pd.read_csv('/content/drive/MyDrive/Dataset/Preprocessed/commercial_group.csv')
dong = pd.read_excel('/content/drive/MyDrive/Dataset/Origin/beopjeongdong_dataset.xlsx')

In [ ]:
admin_mapping = dong[['행정동명', '시군구명']].copy()
admin_mapping.columns = [
    '행정동명',
    '자치구명'
]

admin_mapping = admin_mapping.drop_duplicates()
admin_mapping.head()

,행정동명,자치구명
0,소공동,중구
1,명동,중구
3,남가좌2동,서대문구
4,영등포동,영등포구
5,수유1동,강북구


In [ ]:
commercial = commercial.rename(
    columns={'행정동_코드_명':'행정동명'})

In [ ]:
import re

def normalize_dong(x):
    x = str(x)

    #공백 제거
    x = x.replace(" ", "")

    #구분자 통일
    x = x.replace("·", "")
    x = x.replace(".", "")
    x = x.replace(",", "")

    #괄호 제거
    x = re.sub(r"\(.*?\)", "", x)
    return x

for df in [
    rent_deposit,
    rent_monthly,
    commercial,
    admin_mapping
]:
    df['행정동명'] = (
        df['행정동명']
        .apply(normalize_dong))

In [ ]:
commercial = commercial.merge(
    admin_mapping,
    on='행정동명',
    how='left')

In [ ]:
#전세 데이터
mapping_deposit = (
    rent_deposit
    .merge(
        commercial,
        on='행정동명',
        how='left')
    .merge(
        crime[
            ['자치구명',
             'crime_rate',
             'clearance_rate',
             'crime_grade']],
        on='자치구명',
        how='left'))

In [ ]:
final_deposit.head()

,행정동명,보증금(만원),임대면적(㎡),행정동_코드,medical,education,shopping,bank,gov,culture,intercity,intracity,자치구명,crime_rate,clearance_rate,crime_grade
0,가락2동,65000.0,59.968,11710632.0,17.0,7.0,0.0,6.0,3.0,0.0,0.0,23.0,송파구,0.064020,0.732509,DANGER
1,가리봉동,15000.0,30.330,11530595.0,6.0,2.0,0.0,0.0,0.0,23.0,0.0,8.0,구로구,0.041797,0.729130,NORMAL
2,가산동,20050.0,25.630,11545510.0,22.0,2.0,0.0,25.0,10.0,18.0,0.0,114.0,금천구,0.024957,0.847794,SAFE
3,가양1동,34000.0,49.500,11500603.0,20.0,6.0,0.0,13.0,5.0,3.0,0.0,74.0,강서구,0.049592,0.851048,DANGER
4,가회동,24750.0,43.480,11110600.0,1.0,6.0,0.0,3.0,5.0,0.0,0.0,22.0,종로구,0.034212,1.174684,NORMAL


In [ ]:
#월세 데이터
mapping_monthly = (
    rent_monthly
    .merge(
        commercial,
        on='행정동명',
        how='left')
    .merge(
        crime[
            ['자치구명',
             'crime_rate',
             'clearance_rate',
             'crime_grade']],
        on='자치구명',
        how='left'))

final_monthly.head()

,행정동명,보증금(만원),임대료(만원),임대면적(㎡),행정동_코드,medical,education,shopping,bank,gov,culture,intercity,intracity,자치구명,crime_rate,clearance_rate,crime_grade
0,가락2동,14954.0,61.0,43.070,11710632.0,17.0,7.0,0.0,6.0,3.0,0.0,0.0,23.0,송파구,0.064020,0.732509,DANGER
1,가리봉동,1000.0,47.0,22.500,11530595.0,6.0,2.0,0.0,0.0,0.0,23.0,0.0,8.0,구로구,0.041797,0.729130,NORMAL
2,가산동,1000.0,55.0,17.690,11545510.0,22.0,2.0,0.0,25.0,10.0,18.0,0.0,114.0,금천구,0.024957,0.847794,SAFE
3,가양1동,3000.0,60.0,34.440,11500603.0,20.0,6.0,0.0,13.0,5.0,3.0,0.0,74.0,강서구,0.049592,0.851048,DANGER
4,가회동,4000.0,67.5,43.625,11110600.0,1.0,6.0,0.0,3.0,5.0,0.0,0.0,22.0,종로구,0.034212,1.174684,NORMAL


In [ ]:
mapping_deposit.to_csv('/content/drive/MyDrive/Dataset/Preprocessed/merge_deposit.csv', index=False)
mapping_monthly.to_csv('/content/drive/MyDrive/Dataset/Preprocessed/merge_monthly.csv', index=False)